In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
import matplotlib.pyplot as plt

In [4]:
import seaborn as sns

In [5]:
df = pd.read_csv("C:/Users/hp/Downloads/employee_salary_regression.csv")
print(df)
df.head()

    employee_id  age  years_experience education_level           job_role  \
0       EMP0001   29                 9          Master  Software Engineer   
1       EMP0002   27                 6        Bachelor        ML Engineer   
2       EMP0003   36                13          Master       Data Analyst   
3       EMP0004   43                23     High School             DevOps   
4       EMP0005   24                 1     High School             DevOps   
..          ...  ...               ...             ...                ...   
995     EMP0996   37                13          Master        ML Engineer   
996     EMP0997   27                 5             PhD    Product Manager   
997     EMP0998   33                10        Bachelor  Software Engineer   
998     EMP0999   33                 9     High School             DevOps   
999     EMP1000   40                20        Bachelor             DevOps   

     city_tier  performance_score  num_skills  remote_work  annual_salary_u

,employee_id,age,years_experience,education_level,job_role,city_tier,performance_score,num_skills,remote_work,annual_salary_usd
0,EMP0001,29,9,Master,Software Engineer,1,2.4,3,0,106343.31
1,EMP0002,27,6,Bachelor,ML Engineer,3,2.1,5,1,82852.60
2,EMP0003,36,13,Master,Data Analyst,1,4.1,7,1,142019.59
3,EMP0004,43,23,High School,DevOps,1,3.1,7,1,159972.80
4,EMP0005,24,1,High School,DevOps,1,3.7,12,1,94126.86


In [6]:
df.size

10000

In [7]:
df.shape

(1000, 10)

In [8]:
df.ndim

2

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   employee_id        1000 non-null   object 
 1   age                1000 non-null   int64  
 2   years_experience   1000 non-null   int64  
 3   education_level    1000 non-null   object 
 4   job_role           1000 non-null   object 
 5   city_tier          1000 non-null   int64  
 6   performance_score  1000 non-null   float64
 7   num_skills         1000 non-null   int64  
 8   remote_work        1000 non-null   int64  
 9   annual_salary_usd  1000 non-null   float64
dtypes: float64(2), int64(5), object(3)
memory usage: 78.3+ KB


In [10]:
df.drop(columns =["employee_id"],axis = 1,inplace= True)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                1000 non-null   int64  
 1   years_experience   1000 non-null   int64  
 2   education_level    1000 non-null   object 
 3   job_role           1000 non-null   object 
 4   city_tier          1000 non-null   int64  
 5   performance_score  1000 non-null   float64
 6   num_skills         1000 non-null   int64  
 7   remote_work        1000 non-null   int64  
 8   annual_salary_usd  1000 non-null   float64
dtypes: float64(2), int64(5), object(2)
memory usage: 70.4+ KB


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [13]:
X = df.drop("annual_salary_usd", axis=1)
y = df["annual_salary_usd"]


In [14]:
categorical_cols = ["education_level", "job_role"]

numerical_cols = [
    "age",
    "years_experience",
    "city_tier",
    "performance_score",
    "num_skills",
    "remote_work"
]

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numerical_cols)
    ]
)

In [16]:
from xgboost import XGBRegressor


In [17]:
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [18]:
from sklearn.pipeline import Pipeline

In [19]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['education_level',
                                                   'job_role']),
                                                 ('num', 'passthrough',
                                                  ['age', 'years_experience',
                                                   'city_tier',
                                                   'performance_score',
                                                   'num_skills',
                                                   'remote_work'])])),
                ('model',
                 XGBRegressor(base_score=None, booster=None, callbacks=None,
                              colsample_bylevel=None, colsampl...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [23]:
y_pred = pipeline.predict(X_test)

In [24]:
from sklearn.metrics import r2_score, mean_absolute_error

In [25]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)


In [26]:
print("R2 Score:", round(r2, 4))
print("MAE:", round(mae, 2))

R2 Score: 0.9775
MAE: 4422.49


In [27]:
import joblib

joblib.dump(pipeline, "employee_salary_model.pkl")
print("Model Saved Successfully!")

Model Saved Successfully!


In [28]:

import pandas as pd
import gradio as gr
import joblib
model = joblib.load("employee_salary_model.pkl")
def predict_salary(
    age,
    years_experience,
    education_level,
    job_role,
    city_tier,
    performance_score,
    num_skills,
    remote_work
):

    data = pd.DataFrame({
        "age": [age],
        "years_experience": [years_experience],
        "education_level": [education_level],
        "job_role": [job_role],
        "city_tier": [city_tier],
        "performance_score": [performance_score],
        "num_skills": [num_skills],
        "remote_work": [remote_work]
    })

    prediction = model.predict(data)[0]

    return f"Predicted Annual Salary: ${prediction:,.2f}"

app = gr.Interface(
    fn=predict_salary,
    inputs=[
        gr.Number(label="Age"),
        gr.Number(label="Years of Experience"),
        gr.Dropdown(
            ["High School", "Bachelors", "Masters", "PhD"],
            label="Education Level"
        ),
        gr.Textbox(label="Job Role"),
        gr.Dropdown([1, 2, 3], label="City Tier"),
        gr.Slider(1, 5, step=0.1, label="Performance Score"),
        gr.Number(label="Number of Skills"),
        gr.Radio([0, 1], label="Remote Work (0=No, 1=Yes)")
    ],
    outputs=gr.Textbox(label="Salary Prediction"),
    title="Employee Salary Predictor",
    description="Predict annual salary based on employee details."
)

app.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
